In [ ]:
# Transformaciones avanzadas en dataset de e-commerce

In [ ]:
# Datos base:

In [5]:
import pandas as pd
import numpy as np

# Clientes
clientes = pd.DataFrame({
    'cliente_id': range(1, 6),
    'nombre': ['Ana', 'Juan', 'María', 'Pedro', 'Laura'],
    'segmento': ['Premium', 'Regular', 'Premium', 'Regular', 'VIP']
})

# Pedidos
pedidos = pd.DataFrame({
    'pedido_id': range(1, 11),
    'cliente_id': np.random.choice(range(1, 6), 10),
    'producto': np.random.choice(['A', 'B', 'C', 'D'], 10),
    'precio': np.random.uniform(50, 500, 10).round(2),
    'fecha': pd.date_range('2024-01-01', periods=10)
})

print("Clientes y pedidos cargados")
print("Clientes")
print(clientes)
print("Pedidos")
print(pedidos)


Clientes y pedidos cargados
Clientes
   cliente_id nombre segmento
0           1    Ana  Premium
1           2   Juan  Regular
2           3  María  Premium
3           4  Pedro  Regular
4           5  Laura      VIP
Pedidos
   pedido_id  cliente_id producto  precio      fecha
0          1           2        C  182.47 2024-01-01
1          2           2        A   69.11 2024-01-02
2          3           5        D  495.93 2024-01-03
3          4           5        D  211.08 2024-01-04
4          5           1        A  405.48 2024-01-05
5          6           2        A  484.58 2024-01-06
6          7           2        C   62.72 2024-01-07
7          8           1        B  300.59 2024-01-08
8          9           4        D  474.69 2024-01-09
9         10           2        B   75.94 2024-01-10


In [ ]:
# Enriquecer datos con joins:

In [6]:
# Unir pedidos con información de clientes
pedidos_enriquecidos = pd.merge(
    pedidos, 
    clientes, 
    on='cliente_id', 
    how='left'
)

print("Pedidos con información de clientes:")
print(pedidos_enriquecidos.head())

Pedidos con información de clientes:
   pedido_id  cliente_id producto  precio      fecha nombre segmento
0          1           2        C  182.47 2024-01-01   Juan  Regular
1          2           2        A   69.11 2024-01-02   Juan  Regular
2          3           5        D  495.93 2024-01-03  Laura      VIP
3          4           5        D  211.08 2024-01-04  Laura      VIP
4          5           1        A  405.48 2024-01-05    Ana  Premium


In [ ]:
#  Calcular métricas derivadas:

In [7]:
# Calcular métricas por cliente
metricas_cliente = pedidos_enriquecidos.groupby(['cliente_id', 'nombre', 'segmento']).agg({
    'pedido_id': 'count',
    'precio': ['sum', 'mean', 'max'],
    'fecha': 'max'  # Última compra
}).round(2)

# Aplanar columnas multi-nivel
metricas_cliente.columns = ['num_pedidos', 'total_gastado', 'gasto_promedio', 'gasto_maximo', 'ultima_compra']
metricas_cliente = metricas_cliente.reset_index()

print("\nMétricas por cliente:")
print(metricas_cliente)


Métricas por cliente:
   cliente_id nombre segmento  num_pedidos  total_gastado  gasto_promedio  \
0           1    Ana  Premium            2         706.07          353.04   
1           2   Juan  Regular            5         874.82          174.96   
2           4  Pedro  Regular            1         474.69          474.69   
3           5  Laura      VIP            2         707.01          353.50   

   gasto_maximo ultima_compra  
0        405.48    2024-01-08  
1        484.58    2024-01-10  
2        474.69    2024-01-09  
3        495.93    2024-01-04  


In [ ]:
# Validar reglas de negocio:

In [8]:
def validar_reglas_negocio(df):
    validaciones = []
    
    # VIP deben tener al menos 2 pedidos
    vip_insuficientes = df[(df['segmento'] == 'VIP') & (df['num_pedidos'] < 2)]
    if len(vip_insuficientes) > 0:
        validaciones.append(f"VIPs con pocos pedidos: {len(vip_insuficientes)}")
    
    # Premium no deben exceder gasto máximo
    premium_excesivos = df[(df['segmento'] == 'Premium') & (df['gasto_maximo'] > 800)]
    if len(premium_excesivos) > 0:
        validaciones.append(f"Premiums con gastos excesivos: {len(premium_excesivos)}")
    
    return validaciones

reglas_incumplidas = validar_reglas_negocio(metricas_cliente)
print(f"\nReglas de negocio incumplidas: {reglas_incumplidas}")


Reglas de negocio incumplidas: []


In [ ]:
""" 
¿Qué join usar para mantener todos los registros de la tabla principal?

LEFT JOIN
• Se usa cuando tienes una tabla principal (A) y una tabla secundaria (B).
• El LEFT JOIN devuelve todos los registros de A, incluso si no existe coincidencia en B.
• Los campos de B aparecerán como  cuando no haya match.

¿Cómo decidir qué métricas calcular en un análisis?
Una forma sólida de decidir métricas es seguir un proceso como el siguiente:

1. Define el objetivo del análisis
Antes de pensar en métricas, pregúntarse:
• ¿Qué decisión debe tomar el negocio?
• ¿Qué comportamiento se requiere entender?
• ¿Qué hipótesis se quiere validar?
    Ejemplo:
    • se requiere entender por qué bajaron las ventas en enero.
    • Necesita medir la eficiencia del funnel de conversión.

2. Identifica las entidades y eventos clave
Esto ayudara a saber qué medir:
• Entidades: clientes, productos, pedidos, sesiones.
• Eventos: compra, registro, abandono, clic, devolución.

3. Seleccionar métricas según el tipo de análisis
Descriptivo
    • Totales, promedios, medianas, conteos.
    • Ej: ventas totales, clientes activos, ticket promedio.
Diagnóstico
    • Ratios, tasas, comparaciones, variaciones.
    • Ej: tasa de conversión, churn, crecimiento vs. mes anterior.
Predictivo
    • Tendencias, estacionalidad, cohortes.
    • Ej: LTV proyectado, forecast de demanda.
Operacional
    • Velocidad, eficiencia, tiempos.
    • Ej: tiempo de despacho, SLA cumplidos.

4. Alinear métricas con stakeholders
Una métrica es útil solo si:
• Responde una pregunta del negocio.
• Es accionable.
• Es entendible para quienes toman decisiones.
    Ejemplo:
    • Un CFO quiere márgenes.
    • Un gerente de marketing quiere CAC, ROAS.
    • Un equipo de operaciones quiere tiempos y SLA.

5. Validar disponibilidad y calidad de datos
A veces la métrica ideal no es posible:
• ¿Existe el dato?
• ¿Tiene granularidad suficiente?
• ¿Es confiable?
Si no, definir una metrica sustitutiva (proxy metric).

6. Construir un set mínimo de métricas esenciales
Una buena regla:
• 3 métricas primarias (las que responden la pregunta central).
• 5–7 métricas de soporte (contexto, diagnóstico, segmentación).

"""